In [1]:
import pandas as pd
import os
import numpy as np
import re
from datetime import datetime, timedelta

In [2]:
pd.set_option('display.max_columns', None)

# **Step 1. Merge Cancer patients with EHR**
### output: <span style="color:DodgerBlue;">**cancer_ehr** & **cancer_ehr_@index_1**</span>

#### <span style="color:#FF6347;">**READ**</span> cancer_candidate file

In [3]:
# file_path = "P:/Dataset/MO-DBT-data-curation/Cancer/mo_cancer.xlsx"
file_path = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", "Cancer", "cancer_candidate" + ".xlsx")
df_input = pd.read_excel(file_path)
df = df_input[(df_input["Index"]=="INDEX") & (df_input["Series"]=="DBT")]

In [4]:
df.head(3)

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,EXAM_NAME,Study,Series,BX_ID,PATHOLOGY_DATE,LESION_CLASS,SIDE,Index,time_to_INDEX_days,time_to_INDEX_months,path_to_INDEX_months
2,4330018595,60103700,Heterogeneously dense (51% - 75%),"5 - Highly suggestive of malignancy, appropria...",B-Biopsy should be considered,2020-06-02,DIAG DIG MAMMO RIGHT ALL VIEWS WITH TOMOSYNTHESIS,DIAG,DBT,487690.0,2020-06-05,Malignant,R,INDEX,NaN,NaN,0.1
3,4330018595,60690108,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2020-06-02,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,SCREEN,DBT,NaN,NaN,NaN,NaN,INDEX,NaN,NaN,NaN
4,4330066079,61259016,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2020-01-14,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,DIAG,DBT,NaN,NaN,NaN,L,INDEX,NaN,NaN,NaN


In [5]:
df.columns

Index(['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'COMPOSITION_NAME',
       'FINDING_CATEGORY', 'FINDING_REC', 'EXAM_COMPLETED_DATE', 'EXAM_NAME',
       'Study', 'Series', 'BX_ID', 'PATHOLOGY_DATE', 'LESION_CLASS', 'SIDE',
       'Index', 'time_to_INDEX_days', 'time_to_INDEX_months',
       'path_to_INDEX_months'],
      dtype='object')

In [6]:
extract_cols = ['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'EXAM_COMPLETED_DATE', 'Study', 'Index']

In [7]:
df_patient_visit = df.drop_duplicates(
    subset=extract_cols, keep = 'first').reset_index(drop = True)[extract_cols]

In [8]:
df_patient_visit.head(5)

,PATIENT_STUDY_ID,ACCESSION_NUMBER,EXAM_COMPLETED_DATE,Study,Index
0,4330018595,60103700,2020-06-02,DIAG,INDEX
1,4330018595,60690108,2020-06-02,SCREEN,INDEX
2,4330066079,61259016,2020-01-14,DIAG,INDEX
3,4330303540,73631634,2017-01-10,DIAG,INDEX
4,4330325599,75132824,2016-04-20,DIAG,INDEX


#### <span style="color:#FF6347;">**READ**</span> **EHR** (hormonal_mens, risk_factors, patient_demo, vitals )

In [9]:
hormonal_mens = pd.read_excel(os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", "Cancer", "Cleaned", "hormonal_mens" + ".xlsx"))
risk_factors = pd.read_excel(os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", "Cancer", "Cleaned", "risk_factors" + ".xlsx"))
patient_demo = pd.read_excel(os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", "Cancer", "Cleaned", "patient_demo" + ".xlsx"))
vitals = pd.read_excel(os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", "Cancer", "Cleaned", "vitals" + ".xlsx"))

### **1. Merge BMI from vitals**

In [10]:
vitals.head(5)

,PATIENT_STUDY_ID,DATE_TAKEN,WEIGHT,WEIGHT_UNIT,HEIGHT,HEIGHT_UNIT,BMI
0,4330018595,2019-06-24,2208.0,OZ,63.5,IN,24.06
1,4330018595,2020-03-03,2176.0,OZ,63.5,IN,23.71
2,4330018595,2020-06-23,2000.0,OZ,63.5,IN,21.80
3,4330018595,2020-07-07,2000.0,OZ,63.5,IN,21.80
4,4330018595,2020-08-11,2000.0,OZ,63.5,IN,21.80


In [11]:
df_patient_visit['EXAM_COMPLETED_DATE'] = pd.to_datetime(df_patient_visit['EXAM_COMPLETED_DATE'])
vitals['DATE_TAKEN']                    = pd.to_datetime(vitals['DATE_TAKEN'])

visit_sorted  = df_patient_visit.sort_values('EXAM_COMPLETED_DATE').reset_index(drop=True)
vitals_sorted = vitals.sort_values('DATE_TAKEN').reset_index(drop=True)

df_patient_visit_vital = pd.merge_asof(
    visit_sorted,
    vitals_sorted,
    left_on  = 'EXAM_COMPLETED_DATE',
    right_on = 'DATE_TAKEN',
    by       = 'PATIENT_STUDY_ID',
    # tolerance=pd.Timedelta(days=365),
    direction= 'nearest'
)

print(f"df_patient_visit : {len(df_patient_visit)} rows")
print(f"df_patient_visit_vital : {len(df_patient_visit_vital)} rows")

df_patient_visit : 2436 rows
df_patient_visit_vital : 2436 rows


In [12]:
df_patient_visit_vital.sort_values(['PATIENT_STUDY_ID', 'ACCESSION_NUMBER'], ignore_index=True, inplace=True)

In [13]:
df_patient_visit_vital.head(5)

,PATIENT_STUDY_ID,ACCESSION_NUMBER,EXAM_COMPLETED_DATE,Study,Index,DATE_TAKEN,WEIGHT,WEIGHT_UNIT,HEIGHT,HEIGHT_UNIT,BMI
0,4330018595,60103700,2020-06-02,DIAG,INDEX,2020-06-23,2000.0,OZ,63.5,IN,21.80
1,4330018595,60690108,2020-06-02,SCREEN,INDEX,2020-06-23,2000.0,OZ,63.5,IN,21.80
2,4330066079,61259016,2020-01-14,DIAG,INDEX,2020-07-06,1836.8,OZ,61.0,IN,21.69
3,4330303540,73631634,2017-01-10,DIAG,INDEX,2017-02-03,2480.0,OZ,58.5,IN,31.84
4,4330325599,75132824,2016-04-20,DIAG,INDEX,2016-04-20,2144.0,OZ,65.0,IN,22.30


### **2. Merge hormonal_mens, risk_factors, patient_demo**

In [14]:
merge_hr = hormonal_mens.merge(risk_factors, on=['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'EXAM_COMPLETED_DATE'], how='inner')
merge_hrp = merge_hr.merge(patient_demo, on=['PATIENT_STUDY_ID'], how='inner')

### **2.1. Calculate Age at exam date**

#### <span style="color:darkcyan">**Function**</span>

In [15]:
from pandas import Int64Dtype
def calculate_age_at_study(df):
    """
    Calculates the patient's age in years at the time of the study 
    based on the 'PatientBirthDate' and 'StudyDate' columns.
    """
    
    # 1. Ensure date columns are in datetime format
    # The format 'YYYY-MM-DD' is used for parsing
    try:
        df['BIRTH_DATE_DT'] = pd.to_datetime(df['BIRTH_DATE'], format='%Y-%m-%d')
        # Note: The StudyDate column often includes time (e.g., '2020-06-01 09:10:52').
        # We can let pandas infer the format for this one since it's cleaner.
        df['EXAM_COMPLETED_DATE_DT'] = pd.to_datetime(df['EXAM_COMPLETED_DATE'], errors='coerce') 
    except ValueError as e:
        print(f"Error parsing date format: {e}. Please check your date column formats.")
        return df

    # 2. Calculate the difference in days
    time_difference = df['EXAM_COMPLETED_DATE_DT'] - df['BIRTH_DATE_DT']
    
    # 3. Convert the difference into whole years (integer format)
    # The .dt.days attribute gives the number of days, which is divided by 365.25 
    # and then explicitly cast to an integer to capture only the full years elapsed.
    mask = df['BIRTH_DATE_DT'].notna() & df['EXAM_COMPLETED_DATE_DT'].notna()
    df.loc[mask, 'PatientAge'] = (time_difference[mask].dt.days / 365.25).astype(int)
    # df['PatientAge'] = (time_difference.dt.days / 365.25).astype('Int64')
    
    # Clean up the intermediate columns
    
    df = df.drop(columns=['BIRTH_DATE_DT', 'EXAM_COMPLETED_DATE_DT'])
    
    return df

#### <span style="color:blue">**Main**</span>

In [16]:
merge_hrp = calculate_age_at_study(merge_hrp)

In [17]:
merge_hrp.head(3)

,PATIENT_STUDY_ID,ACCESSION_NUMBER,AGE_MENARCHE,AGE_FIRST_LIVE_BIRTH,AGE_MENOPAUSE,AGE_HYSTERECTOMY,AGE_RIGHT_OVARY_REMOVAL,AGE_LEFT_OVARY_REMOVAL,PARITY_COUNT,PREGNANCY_COUNT,LAST_MENSTRUAL_DATE,MENSTRUAL_STATUS_CD,EXAM_COMPLETED_DATE,RISK_FACTOR_No family history of breast cancer,RISK_FACTOR_Personal breast cancer history,"RISK_FACTOR_Weak family history of breast cancer (aunt, grandmother, cousin)",RISK_FACTOR_Late child bearing (after 30),"RISK_FACTOR_Very strong family history of breast cancer (mother, sister, daughter, pre-menopause or multiple post-menopausal first degree relatives)","RISK_FACTOR_Family history of ovarian cancer in mother, sister, or daughter",RISK_FACTOR_Post-menopausal patient,RISK_FACTOR_BRCA1 gene mutation,RISK_FACTOR_Nulliparous,RISK_FACTOR_Family history of ovarian cancer in distant blood relatives,"RISK_FACTOR_Intermediate family history of breast cancer (mother, sister, daughter, post-menopause)","RISK_FACTOR_History of high risk lesion on previous biopsy, LCIS, atypical hyperplasia",RISK_FACTOR_Previous chest radiation therapy,RISK_FACTOR_Other family history of breast cancer,RISK_FACTOR_BRCA2 gene mutation,RISK_FACTOR_Family history unknown for breast cancer,RISK_FACTOR_History of ovarian cancer,RISK_FACTOR_History of endometrial cancer,RISK_FACTOR_Not available,BIRTH_DATE,GENDER_TITLE,RACE_KOREAN,RACE_WHITE,RACE_INDIAN (ASIAN),RACE_BLACK,RACE_UNREPORTED/CHOSE NOT TO DISCLOSE,RACE_OTHER ASIAN,RACE_VIETNAMESE,RACE_CHINESE,RACE_AMERICAN INDIAN/ALASKA NATIVE,RACE_FILIPINO,RACE_OTHER,RACE_JAPANESE,RACE_HAWAIIAN,RACE_OTHER PACIFIC ISLANDER,ETHNIC_NOT HISPANIC OR LATINO,"ETHNIC_ANOTHER HISPANIC, LATINO, OR SPANISH ORIGIN",ETHNIC_UNREPORTED/CHOSE NOT TO DISCLOSE,ETHNIC_HISPANIC OR LATINO,"ETHNIC_MEXICAN, MEXICAN AMERICAN, CHICANO",ETHNIC_PUERTO RICAN,PatientAge
0,4330018595,63027507,16,31,49,0,0,0,2.0,3.0,NaN,NaN,2019-07-26,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1965-07-01,FEMALE,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,54.0
1,4330018595,63737104,16,31,49,0,0,0,2.0,3.0,NaN,NaN,2019-08-19,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1965-07-01,FEMALE,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,54.0
2,4330018595,60103700,16,31,49,0,0,0,2.0,3.0,NaN,NaN,2020-06-02,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1965-07-01,FEMALE,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,54.0


### **3. Merge MO cancer patients with EHR**

In [18]:
merge_hrp['EXAM_COMPLETED_DATE'] = pd.to_datetime(merge_hrp['EXAM_COMPLETED_DATE'])

In [19]:
df_patient_visit_ehr = pd.merge(df_patient_visit_vital, merge_hrp, on=['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'EXAM_COMPLETED_DATE'], how='left')

In [20]:
df_patient_visit_ehr

,PATIENT_STUDY_ID,ACCESSION_NUMBER,EXAM_COMPLETED_DATE,Study,Index,DATE_TAKEN,WEIGHT,WEIGHT_UNIT,HEIGHT,HEIGHT_UNIT,BMI,AGE_MENARCHE,AGE_FIRST_LIVE_BIRTH,AGE_MENOPAUSE,AGE_HYSTERECTOMY,AGE_RIGHT_OVARY_REMOVAL,AGE_LEFT_OVARY_REMOVAL,PARITY_COUNT,PREGNANCY_COUNT,LAST_MENSTRUAL_DATE,MENSTRUAL_STATUS_CD,RISK_FACTOR_No family history of breast cancer,RISK_FACTOR_Personal breast cancer history,"RISK_FACTOR_Weak family history of breast cancer (aunt, grandmother, cousin)",RISK_FACTOR_Late child bearing (after 30),"RISK_FACTOR_Very strong family history of breast cancer (mother, sister, daughter, pre-menopause or multiple post-menopausal first degree relatives)","RISK_FACTOR_Family history of ovarian cancer in mother, sister, or daughter",RISK_FACTOR_Post-menopausal patient,RISK_FACTOR_BRCA1 gene mutation,RISK_FACTOR_Nulliparous,RISK_FACTOR_Family history of ovarian cancer in distant blood relatives,"RISK_FACTOR_Intermediate family history of breast cancer (mother, sister, daughter, post-menopause)","RISK_FACTOR_History of high risk lesion on previous biopsy, LCIS, atypical hyperplasia",RISK_FACTOR_Previous chest radiation therapy,RISK_FACTOR_Other family history of breast cancer,RISK_FACTOR_BRCA2 gene mutation,RISK_FACTOR_Family history unknown for breast cancer,RISK_FACTOR_History of ovarian cancer,RISK_FACTOR_History of endometrial cancer,RISK_FACTOR_Not available,BIRTH_DATE,GENDER_TITLE,RACE_KOREAN,RACE_WHITE,RACE_INDIAN (ASIAN),RACE_BLACK,RACE_UNREPORTED/CHOSE NOT TO DISCLOSE,RACE_OTHER ASIAN,RACE_VIETNAMESE,RACE_CHINESE,RACE_AMERICAN INDIAN/ALASKA NATIVE,RACE_FILIPINO,RACE_OTHER,RACE_JAPANESE,RACE_HAWAIIAN,RACE_OTHER PACIFIC ISLANDER,ETHNIC_NOT HISPANIC OR LATINO,"ETHNIC_ANOTHER HISPANIC, LATINO, OR SPANISH ORIGIN",ETHNIC_UNREPORTED/CHOSE NOT TO DISCLOSE,ETHNIC_HISPANIC OR LATINO,"ETHNIC_MEXICAN, MEXICAN AMERICAN, CHICANO",ETHNIC_PUERTO RICAN,PatientAge
0,4330018595,60103700,2020-06-02,DIAG,INDEX,2020-06-23,2000.0,OZ,63.5,IN,21.80,16.0,31.0,49.0,0.0,0.0,0.0,2.0,3.0,NaN,NaN,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1965-07-01,FEMALE,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,54.0
1,4330018595,60690108,2020-06-02,SCREEN,INDEX,2020-06-23,2000.0,OZ,63.5,IN,21.80,16.0,31.0,49.0,0.0,0.0,0.0,2.0,3.0,NaN,NaN,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1965-07-01,FEMALE,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,54.0
2,4330066079,61259016,2020-01-14,DIAG,INDEX,2020-07-06,1836.8,OZ,61.0,IN,21.69,14.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2020-07-04,NaN,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1989-07-01,FEMALE,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,30.0
3,4330303540,73631634,2017-01-10,DIAG,INDEX,2017-02-03,2480.0,OZ,58.5,IN,31.84,13.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1937-07-01,FEMALE,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,79.0
4,4330325599,75132824,2016-04-20,DIAG,INDEX,2016-04-20,2144.0,OZ,65.0,IN,22.30,13.0,21.0,0.0,0.0,0.0,0.0,2.0,3.0,NaN,NaN,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1977-07-01,FEMALE,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,38.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2431,4339604670,67519387,2020-12-15,DIAG,INDEX,2020-12-31,2752.0,OZ,61.0,IN,32.50,13.0,0.0,58.0,0.0,58.0,0.0,0.0,1.0,NaN,POSTSUR,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1954-07-01,FEMALE,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,66.0
2432,4339604670,68584905,2020-11-27,SCREEN,

In [21]:
df_patient_visit_ehr.sort_values(['PATIENT_STUDY_ID', 'EXAM_COMPLETED_DATE', 'ACCESSION_NUMBER'], ignore_index=True, inplace=True)

#### <span style="color:#FF6347;">**SAVE**</span> **cancer_ehr, cancer_ehr_unique_patient**

In [22]:
df_patient_visit_ehr["DATE_TAKEN"] = df_patient_visit_ehr["DATE_TAKEN"].dt.strftime("%Y-%m-%d")
df_patient_visit_ehr["EXAM_COMPLETED_DATE"] = df_patient_visit_ehr["EXAM_COMPLETED_DATE"].dt.strftime("%Y-%m-%d")

In [23]:
df_patient_visit_ehr['PATIENT_STUDY_ID'].nunique(), df_patient_visit_ehr.shape

(1630, (2436, 63))

In [24]:
df_patient_ehr = df_patient_visit_ehr.drop_duplicates(subset=["PATIENT_STUDY_ID"], keep="first")

In [25]:
df_patient_ehr['PATIENT_STUDY_ID'].nunique(), df_patient_ehr.shape

(1630, (1630, 63))

In [26]:
# output_file = os.path.join("P:/Dataset/MO-DBT-data-curation/", "Cancer", 'mo_cancer_ehr' + ".xlsx")
# df_patient_visit_ehr.to_excel(output_file, index=False)

# output_file = os.path.join("P:/Dataset/MO-DBT-data-curation/", "Cancer", 'mo_cancer_ehr_@_index_1' + ".xlsx")
# df_patient_ehr.to_excel(output_file, index=False)


output_file = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", "Cancer", 'cancer_candidate_ehr' + ".xlsx")
df_patient_visit_ehr.to_excel(output_file, index=False)

output_file = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", "Cancer", 'cancer_candidate_ehr_unique_patient' + ".xlsx")
df_patient_ehr.to_excel(output_file, index=False)